# Ablation study

Гоняет `fp2mp_core` на одной задаче с разными `AblationConfig`, чтобы измерить вклад каждого компонента (ReDI / BlackBoard / Critic / Tools).

**Что меняется в блокноте** (всё остальное вынесено в `notebook_lib.py`):

- `LLM_PROVIDER` — `"ollama"` (default) или `"openai_compat"`.
- `MODEL`, `MAX_ITER`, `QUESTION`.
- `SCENARIOS` — какие из пяти базовых сценариев запускать.

**Что делается автоматически:**

- Применяется ENV-пресет провайдера (`FP2MP_CHAT_URL`, ключ из `OLLAMA_API_KEY` → `FP2MP_OLLAMA_KEY` → `FP2MP_API_KEY`).
- Каждый сценарий пишет `examples/ablation/runs/<scenario>/result.json` (та же схема, что и CLI-прогоны).
- В конце — Markdown-сводка с метриками и путями к JSON.

Запуск: откройте ноутбук из корня репо (`jupyter lab`) и выполните ячейки сверху вниз.


In [1]:
# --- INPUTS: меняйте под свою задачу/провайдер ---

# "ollama" | "openai_compat"
LLM_PROVIDER = "ollama"

# Имя модели, которую должен поддерживать выбранный провайдер.
# Примеры:
#   ollama:        "ministral-3:8b", "qwen3-coder-next", "deepseek-v4-flash"
#   openai_compat: "gpt-4o-mini", "gpt-4o", "openai/gpt-4.1"
MODEL = "deepseek-v4-pro"

MAX_ITER = 3

# Задача, на которой сравниваем сценарии.
QUESTION = "Создай мастер-план города Гатчины в Ленинградской области в России"

# Какие сценарии гонять. None = все пять.
# Допустимо: "baseline", "redi_off", "blackboard_off", "critic_off", "tools_off".
SCENARIOS = ["blackboard_off", "critic_off", "tools_off"] # например: ["baseline", "tools_off"]


In [2]:
import sys
from pathlib import Path

# Подключаем src/ корень репо (нужно, если Jupyter стартован не из корня).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ablation":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
_SRC = PROJECT_ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
_NB = PROJECT_ROOT / "examples" / "notebook" / "ablation"
if str(_NB) not in sys.path:
    sys.path.insert(0, str(_NB))

from notebook_lib import apply_provider_preset, scenarios_to_run, run_scenarios, build_comparison_table

PROVIDER_CONFIG = apply_provider_preset(LLM_PROVIDER)
_src = PROVIDER_CONFIG["applied"].get("FP2MP_API_KEY_SOURCE", "ENV/.env")
print(
    f"LLM_PROVIDER={LLM_PROVIDER} | MODEL={MODEL} | MAX_ITER={MAX_ITER}\n"
    f"FP2MP_CHAT_URL={PROVIDER_CONFIG['applied'].get('FP2MP_CHAT_URL')} | "
    f"FP2MP_API_KEY source: {_src}"
)


LLM_PROVIDER=ollama | MODEL=deepseek-v4-pro | MAX_ITER=3
FP2MP_CHAT_URL=https://ollama.com/v1 | FP2MP_API_KEY source: FP2MP_API_KEY


In [3]:
SELECTED = scenarios_to_run(SCENARIOS)
print("Scenarios:", list(SELECTED))
print()

SUMMARIES = run_scenarios(
    question=QUESTION,
    model=MODEL,
    max_iterations=MAX_ITER,
    scenarios=SELECTED,
)


Scenarios: ['blackboard_off', 'critic_off', 'tools_off']



/root/workflow/projects/ITMO/fp2mp-core/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



>> blackboard_off  flags={'redi': True, 'blackboard': False, 'critic': True, 'tools': True}


/root/workflow/projects/ITMO/fp2mp-core/src/fp2mp_core/tools/tool_registry.py:54: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6928.13it/s]


   -> /root/workflow/projects/ITMO/fp2mp-core/examples/ablation/runs/blackboard_off/result.json  (598.1s)  answer_len=7837  usage={'input_tokens': 248685, 'output_tokens': 55640, 'total_tokens': 304325, 'source': 'real'}

>> critic_off  flags={'redi': True, 'blackboard': True, 'critic': False, 'tools': True}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6655.13it/s]


   -> /root/workflow/projects/ITMO/fp2mp-core/examples/ablation/runs/critic_off/result.json  (927.9s)  answer_len=5823  usage={'input_tokens': 247782, 'output_tokens': 99279, 'total_tokens': 347061, 'source': 'real'}

>> tools_off  flags={'redi': True, 'blackboard': True, 'critic': True, 'tools': False}
   -> /root/workflow/projects/ITMO/fp2mp-core/examples/ablation/runs/tools_off/result.json  (374.8s)  answer_len=8829  usage={'input_tokens': 26640, 'output_tokens': 33191, 'total_tokens': 59831, 'source': 'real'}


In [4]:
from IPython.display import Markdown, display

display(Markdown("## Ablation comparison\n" + build_comparison_table(SUMMARIES)))

print()
print("Per-scenario artifacts:")
for s in SUMMARIES:
    name = s.get("scenario", "?")
    res = PROJECT_ROOT / "examples" / "ablation" / "runs" / name / "result.json"
    log = PROJECT_ROOT / "examples" / "ablation" / "runs" / name / "log.json"
    print(f"  {name:18s} result={res}")
    if log.exists():
        # log.json большой — только размер и synthesis_sig
        import json as _json
        with log.open() as _f:
            _meta = _json.load(_f)
        print(f"  {'':18s} log   ={log} ({log.stat().st_size // 1024} KB, "
              f"{len(_meta.get('log', []))} entries, sig={_meta.get('synthesis_sig')})")
    else:
        print(f"  {'':18s} log   =(not written; run failed or summary missing)")


## Ablation comparison
| scenario | flags (off) | status | elapsed | answer len | log entries | input tok | output tok | total tok | error |
|---|---|---|---:|---:|---:|---:|---:|---:|---|
| blackboard_off | blackboard | ✅ | 598.1 | 7837 | 145 | 248685 | 55640 | 304325 |  |
| critic_off | critic | ✅ | 927.9 | 5823 | 89 | 247782 | 99279 | 347061 |  |
| tools_off | tools | ✅ | 374.8 | 8829 | 20 | 26640 | 33191 | 59831 |  |


Per-scenario artifacts:
  blackboard_off     result=/root/workflow/projects/ITMO/fp2mp-core/examples/examples/ablation/runs/blackboard_off/result.json
                     log   =(not written; run failed or summary missing)
  critic_off         result=/root/workflow/projects/ITMO/fp2mp-core/examples/examples/ablation/runs/critic_off/result.json
                     log   =(not written; run failed or summary missing)
  tools_off          result=/root/workflow/projects/ITMO/fp2mp-core/examples/examples/ablation/runs/tools_off/result.json
                     log   =(not written; run failed or summary missing)
